In [ ]:
import os
import json
import glob
import torchaudio
import torch

import numpy as np
from torch.utils.data import Dataset, DataLoader

In [ ]:
# Mapping STIM_TYPE to dataset attributes
STIMULUS_CONFIG = {
    "NHS vs Global-Music": {
        "representation_folder": "music",
        "sound_list": "/om2/user/bjmedina/BOLIVIA2024/assets/music/soundlist.csv",
        "stim_filenames": "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_nhs-global_p1/filenames.json",
    },
    "nhs": {
        "representation_folder": "music",
        "sound_list": "/om2/user/bjmedina/BOLIVIA2024/assets/music/soundlist.csv",
        "stim_filenames": "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_nhs-global_p1/filenames.json",
    },
    "global-music": {
        "representation_folder": "music",
        "sound_list": "/om2/user/bjmedina/BOLIVIA2024/assets/music/soundlist.csv",
        "stim_filenames": "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_nhs-global_p1/filenames.json",
    },
    "Auditory Textures": {
        "representation_folder": "textures",
        "sound_list": "/om2/user/bjmedina/BOLIVIA2024/assets/textures/textures.csv",
        "stim_filenames": "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_p1/filenames.json",
    },
    "Industrial vs Nature": {
        "representation_folder": "environmental",
        "sound_list": "/om2/user/bjmedina/BOLIVIA2024/assets/environmental/soundlist.csv",
        "stim_filenames": "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_ind-nature_p1/filenames.json",
    },
    "industrial": {
        "representation_folder": "environmental",
        "sound_list": "/om2/user/bjmedina/BOLIVIA2024/assets/environmental/soundlist.csv",
        "stim_filenames": "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_ind-nature_p1/filenames.json",
    },
    "nature": {
        "representation_folder": "environmental",
        "sound_list": "/om2/user/bjmedina/BOLIVIA2024/assets/environmental/soundlist.csv",
        "stim_filenames": "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_ind-nature_p1/filenames.json",
    },
}


class MemoryDataset(Dataset):
    def __init__(self, representation_dir, stim_filenames, transform=None, sample_rate=44100):
        """
        Args:
            representation_dir (str): Path to the directory particular representation for each sound.
            stim_filenames (str): Path to JSON file containing stimulus filenames.
            transform (callable, optional): Optional transform to be applied.
            sample_rate (int): Target sample rate for audio files.
        """

        # matrix that contains representation of 
        self.representation_dir = representation_dir
        self.sample_rate = sample_rate
        self.transform = transform

        self.stim_metadata = None
        # Load stimulus filenames from JSON
        with open(stim_filenames, 'r') as f:
            self.stim_metadata = json.load(f)

        self.representation = np.load(self.representation_dir)
        self.stimulus_type  = []
        self.source_path    = []
        self.stim_path      = []

        for stim_metadatum in self.stim_metadata:
            try:
                self.stimulus_type.append(stim_metadatum['type'])
            except KeyError: 
                self.stimulus_type.append('texture')
            self.source_path.append(stim_metadatum['orig_path'])
            self.stim_path.append(stim_metadatum['stim_path'])

    def __str__(self):
        return f"root_dir: {self.representation_dir}\n files: {self.stim_metadata}"

    def __rep__(self):
        return f"root_dir: {self.representation_dir}\n files: {self.stim_metadata}"

    def __len__(self):
        return len(self.stim_path)

    def __getitem__(self, idx):
        file_path = self.stim_path[idx]
        waveform, sr = torchaudio.load(file_path)

        # Convert to mono if it's stereo
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)  # Average across channels

        # Resample if needed
        if sr != self.sample_rate:
            waveform = torchaudio.transforms.Resample(sr, self.sample_rate)(waveform)

        if self.transform:
            waveform = self.transform(waveform)

        return waveform, self.representation[idx], self.stim_path[idx]


def get_dataloader(STIM_TYPE, batch_size=8, shuffle=True, sample_rate=16000):
    """
    Given a stimulus set name (STIM_TYPE), return a DataLoader.

    Args:
        STIM_TYPE (str): Name of the stimulus set.
        batch_size (int): Number of samples per batch.
        shuffle (bool): Whether to shuffle the dataset.
        sample_rate (int): Target sample rate.

    Returns:
        DataLoader: PyTorch DataLoader for the specified audio stimulus set.
    """
    if STIM_TYPE not in STIMULUS_CONFIG:
        raise ValueError(f"Unknown STIM_TYPE '{STIM_TYPE}'. Choose from {list(STIMULUS_CONFIG.keys())}")

    config = STIMULUS_CONFIG[STIM_TYPE]

    dataset_path = glob.glob(f"/om2/user/bjmedina/BOLIVIA2024/assets/{config['representation_folder']}/*npy")[0]
    stim_filenames = config["stim_filenames"]

    print(dataset_path, stim_filenames)
    dataset = MemoryDataset(dataset_path, stim_filenames, sample_rate=sample_rate)

    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


# Example usage
STIM_TYPE = "Auditory Textures"
dataloader = get_dataloader(STIM_TYPE, batch_size=16, shuffle=True, sample_rate=44100)

print(len(dataloader))
for batch in dataloader:
    print()

In [ ]:
dataset_path = glob.glob(f"/om2/user/bjmedina/BOLIVIA2024/assets/music/*npy")[0]
representation = np.load(dataset_path)
print(representation[0].shape)